# Validating the HUXt output

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from storm_utils.config_paths import get_project_paths
import storm_utils.huxt_utils as HU

paths = get_project_paths()

In [ ]:
print("HUXt Utilities - Example Usage\n")
huxt_run_id = 3
# Get run info
info = HU.get_huxt_run_info(huxt_run_id=huxt_run_id)

# Validate
validation = HU.validate_huxt_run(huxt_run_id=huxt_run_id, expected_n_ensemble=2000)

if validation['valid']:
    print("✓ HUXt run is valid and complete!")
else:
    print("✗ Issues found - see details above")

# Plot single rotation
HU.plot_huxt_ensemble(huxt_run_id=huxt_run_id, n_members_to_plot=100)

# Plot multiple rotations
HU.plot_full_huxt_timeseries(huxt_run_id=huxt_run_id, max_rotations=5)

# Detect and save discontinuities
discontinuities = HU.detect_all_discontinuities(huxt_run_id=huxt_run_id, threshold=50, save=True)

# Combine and validate processed HUXt

In [ ]:
from storm_utils.data_processing import process_huxt

# Confirm before running
response = input("\nProceed with HUXt processing? (yes/no): ")
if response.lower() != 'yes':
    print("Run cancelled")
else:
    results = process_huxt(
            huxt_run_id=3,
            chunk_size=20,
            save_discontinuities=True,
            delete_intermediates=False,
            spinup_days=1,
    )

## Validate compatibility with ForecastingDataset

In [ ]:
from storm_utils.data_structure import ForecastingDataset
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()
huxt_id = 3

# Create dataset with the processed file
dataset = ForecastingDataset(
    parquet_path=str(paths['huxt_data_shared'] / f'HUXt{huxt_id}_modified' / 'full_df.parquet'),
    discontinuity_path=str(paths['huxt_data_shared'] / f'HUXt{huxt_id}_modified' / 'discontinuities.npy'),
    seed=42,
    Nens=100,
    lead_time_hours=12,
    forecast_duration_hours=24,
    stride_hours=24
)

In [ ]:
dataset.describe()

In [ ]:
## Available functions for checking ForecastingDataset are: 
# ForecastingDataset.describe()
# ForecastingDataset.plot_dataset_overview()
# ForecastingDataset.plot_reduced_dataset_overview()
# ForecastingDataset.get_storm_statistics()

dataset.plot_dataset_overview()

In [ ]:
## Check no discontinuities are present
from storm_regression.plotting import plot_window_data

print('Check no discontinuities in these windows')
for i in [2500]:
    plot_window_data(dataset, i)

In [ ]:
## Inspect a single processed df
import pandas as pd

paths = get_project_paths()
filepath = paths['huxt_data_shared'] / 'HUXt3_modified' / 'HUXt_rotation_1892.parquet'
single_df = pd.read_parquet(filepath)
single_df

In [ ]:
from scipy.ndimage import maximum_filter1d, minimum_filter1d
from numpy.lib.stride_tricks import sliding_window_view
from storm_utils.data_processing import load_full_omni
import numpy as np

df = load_full_omni()

# ===== SIR Flag =====
verbose = True
v_sw = df['V_sw'].values

# Check for issues
v_sw = np.nan_to_num(v_sw, nan=0.0)

# Calculate window size
sir_time_window_hours = 24
sir_velocity_threshold = 150
time_delta_hours = (df.index[1] - df.index[0]).total_seconds() / 3600
n_points = int(sir_time_window_hours / time_delta_hours)
window_size = min(n_points * 2 + 1, len(v_sw))

# Initialize SIR flag
sir_flag = np.zeros(len(v_sw), dtype=int)

# For each timestep, check the window around it
half_window = window_size // 2

for i in range(half_window, len(v_sw) - half_window):
    # Extract window
    window_start = i - half_window
    window_end = i + half_window + 1
    v_window = v_sw[window_start:window_end]
    
    # Find min and max positions in window
    idx_max_in_window = np.argmax(v_window)
    idx_min_in_window = np.argmin(v_window)
    
    # Convert to absolute indices
    idx_max_abs = window_start + idx_max_in_window
    idx_min_abs = window_start + idx_min_in_window
    
    # Condition 1: Velocity range > threshold
    v_max = v_window[idx_max_in_window]
    v_min = v_window[idx_min_in_window]
    v_range = v_max - v_min
    
    # Condition 2: Max occurs AFTER min (positive gradient)
    # Condition 3: Current timestep is between min and max (inclusive)
    if v_range > sir_velocity_threshold and idx_max_abs > idx_min_abs:
        if idx_min_abs <= i <= idx_max_abs:
            sir_flag[i] = 1

df["SIR_flag"] = sir_flag

if verbose:
    print(f"  SIR flagged: {df['SIR_flag'].sum()} timesteps ({df['SIR_flag'].sum()/len(df)*100:.1f}%)")

import matplotlib.pyplot as plt
import numpy as np

def plot_sir_regions(df, n_regions=5, window_hours=48):
    """
    Plot regions where SIR flags are present to verify detection logic.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with SIR_flag and V_sw columns
    n_regions : int
        Number of SIR regions to plot
    window_hours : int
        Hours to show around each SIR (for context)
    """
    # Find SIR transitions (where flag goes from 0 to 1)
    sir_flag = df['SIR_flag'].values
    sir_starts = []
    
    for i in range(1, len(sir_flag)):
        if sir_flag[i] == 1 and sir_flag[i-1] == 0:
            sir_starts.append(i)
    
    print(f"Found {len(sir_starts)} SIR start events")
    
    if len(sir_starts) == 0:
        print("No SIR events found!")
        return
    
    # Plot first n_regions
    n_to_plot = min(n_regions, len(sir_starts))
    
    # Calculate window size
    time_delta_hours = (df.index[1] - df.index[0]).total_seconds() / 3600
    n_points_context = int(window_hours / time_delta_hours)
    
    fig, axes = plt.subplots(n_to_plot, 1, figsize=(14, 4*n_to_plot))
    if n_to_plot == 1:
        axes = [axes]
    
    for plot_idx, sir_start in enumerate(sir_starts[:n_to_plot]):
        ax = axes[plot_idx]
        
        # Define window around SIR
        window_start = max(0, sir_start - n_points_context)
        window_end = min(len(df), sir_start + n_points_context)
        
        # Extract data
        times = df.index[window_start:window_end]
        v_window = df['V_sw'].values[window_start:window_end]
        sir_flags = df['SIR_flag'].values[window_start:window_end]
        
        # Plot velocity
        ax.plot(times, v_window, 'b-', linewidth=1.5, label='V_sw')
        
        # Plot SIR flag as shaded regions
        sir_mask = sir_flags.astype(bool)
        ax.fill_between(times, ax.get_ylim()[0], ax.get_ylim()[1], where=sir_mask, 
                        alpha=0.3, color='red', label='SIR flag')
        
        # Mark the detection point
        ax.axvline(df.index[sir_start], color='red', linestyle='--', 
                  linewidth=2, label='SIR start', alpha=0.7)
        
        # Find full SIR period
        sir_region_start = sir_start
        sir_region_end = sir_start
        
        # Extend to find full SIR period
        while sir_region_end < len(sir_flag) and sir_flag[sir_region_end] == 1:
            sir_region_end += 1
        
        # Get velocity in SIR region
        v_sir = df['V_sw'].values[sir_region_start:sir_region_end]
        
        if len(v_sir) > 0:
            v_min_val = np.min(v_sir)
            v_max_val = np.max(v_sir)
            v_min_idx = sir_region_start + np.argmin(v_sir)
            v_max_idx = sir_region_start + np.argmax(v_sir)
            
            # Mark min and max
            ax.scatter([df.index[v_min_idx]], [v_min_val], 
                      color='blue', s=100, marker='v', label=f'Min: {v_min_val:.1f}', zorder=5)
            ax.scatter([df.index[v_max_idx]], [v_max_val], 
                      color='green', s=100, marker='^', label=f'Max: {v_max_val:.1f}', zorder=5)
            
            # Annotate with range and gradient direction
            v_range = v_max_val - v_min_val
            gradient_ok = v_max_idx > v_min_idx
            
            ax.text(0.02, 0.98, 
                   f'ΔV = {v_range:.1f} km/s\n'
                   f'Gradient: {"✓ Positive" if gradient_ok else "✗ Negative"}\n'
                   f'Duration: {sir_region_end - sir_region_start} timesteps',
                   transform=ax.transAxes, fontsize=10, 
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        ax.set_ylabel('V_sw (km/s)', fontsize=11, fontweight='bold')
        ax.set_xlabel('Time', fontsize=11)
        ax.set_title(f'SIR Event #{plot_idx + 1} - {df.index[sir_start].strftime("%Y-%m-%d %H:%M")}', 
                    fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(alpha=0.3)
        
        # Format x-axis
        ax.tick_params(axis='x', rotation=45)
        
        # Set y-limits to show all data without cutoff
        y_min = np.min(v_window)
        y_max = np.max(v_window)
        y_range = y_max - y_min
        ax.set_ylim(y_min - 0.1 * y_range, y_max + 0.1 * y_range)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print(f"\n{'='*80}")
    print(f"SIR DETECTION SUMMARY")
    print(f"{'='*80}")
    
    # Check all SIR regions for gradient
    n_positive_gradient = 0
    n_negative_gradient = 0
    
    for sir_start in sir_starts:
        sir_region_end = sir_start
        while sir_region_end < len(sir_flag) and sir_flag[sir_region_end] == 1:
            sir_region_end += 1
        
        v_sir = df['V_sw'].values[sir_start:sir_region_end]
        if len(v_sir) > 0:
            if np.argmax(v_sir) > np.argmin(v_sir):
                n_positive_gradient += 1
            else:
                n_negative_gradient += 1
    
    print(f"Total SIR events: {len(sir_starts)}")
    print(f"  Positive gradient: {n_positive_gradient} ({n_positive_gradient/len(sir_starts)*100:.1f}%)")
    print(f"  Negative gradient: {n_negative_gradient} ({n_negative_gradient/len(sir_starts)*100:.1f}%)")
    print(f"{'='*80}\n")


# Usage:
plot_sir_regions(df, n_regions=10, window_hours=48)